In [1]:
import os

In [2]:
%pwd

'c:\\Users\\sagal\\Desktop\\Let us build\\deforestation_detection\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\Users\\sagal\\Desktop\\Let us build\\deforestation_detection'

entity->configuration manager -> components ->pipeline this is the standard workflow

In [5]:
# 1. Entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path

In [6]:
from deforestation_detection.constant import *
from deforestation_detection.utils.common import read_yaml, create_directories

In [7]:
#2. Configuration Manager 

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_url = config.source_url,
            local_data_file = config.local_data_file,
        )

        return data_ingestion_config

In [22]:
import os
import urllib.request as request
from deforestation_detection.logging import logger
from deforestation_detection.utils.common import get_size
from huggingface_hub import hf_hub_download

#3. Components

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    def download_files(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_url,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} Downloaded with following info \n{headers}")
        else:
            logger.info(f"file already exist of size: {get_size(Path(self.config.local_data_file))}")

#3. components updated

class DataIngestion:
    def __init__(self, config):
        self.config = config

    def download_files(self):
        """
        Downloads the dataset directly using standard urllib request.
        """
        if not os.path.exists(self.config.local_data_file):
            logger.info(f"Downloading dataset from {self.config.source_url} to {self.config.local_data_file}")
            
            # Create a basic request with a browser User-Agent
            req = request.Request(
                self.config.source_url,
                headers={'User-Agent': 'Mozilla/5.0'}
            )
            
            # Direct download and write to file
            with request.urlopen(req) as response, open(self.config.local_data_file, 'wb') as out_file:
                out_file.write(response.read())
                
            logger.info(f"File downloaded successfully to {self.config.local_data_file} with size: {get_size(Path(self.config.local_data_file))}")
        else:
            logger.info(f"File already exists with size: {get_size(Path(self.config.local_data_file))}")

In [ ]:
#3. components
import os
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download
from deforestation_detection.logging import logger
from deforestation_detection.utils.common import get_size

class DataIngestion:
    def __init__(self, config):
        self.config = config

    def download_files(self):
        """
        Downloads the dataset file from Hugging Face using huggingface_hub 
        with explicit anonymous access to handle Xet/LFS pointers.
        """
        if not os.path.exists(self.config.local_data_file):
            try:
                logger.info(f"Downloading dataset from Hugging Face into {self.config.local_data_file}")
                
                # Use hf_hub_download with token=False for public Xet-backed files
                downloaded_path = hf_hub_download(
                    repo_id="Revanth214/Deforestation",
                    filename="dataset_arrays.pt",
                    repo_type="dataset",
                    token=False
                )
                
                os.makedirs(os.path.dirname(self.config.local_data_file), exist_ok=True)
                shutil.copy(downloaded_path, self.config.local_data_file)
                
                logger.info(f"Successfully downloaded file with size: {get_size(Path(self.config.local_data_file))}")
                
            except Exception as e:
                raise e
        else:
            logger.info(f"File already exists with size: {get_size(Path(self.config.local_data_file))}")

In [36]:
#4. Pipeline

try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config = data_ingestion_config)
    data_ingestion.download_files()
except Exception as e:
    raise e

[2026-10-21 17:41:28,089: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-10-21 17:41:28,090: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-10-21 17:41:28,091: INFO: common: created directory at artifacts]
[2026-10-21 17:41:28,092: INFO: common: created directory at artifacts/data_ingestion]
[2026-10-21 17:41:28,092: INFO: 157515968: File already exists with size: ~ 184322 KB]
